<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- **Analysis unit:** one pseudonymized `client_hash_id` × `content_hash_id` candidate, aggregated across its eligible daily records in March 2026.
- **Raw-table grain:** one `report_date` × `client_hash_id` × `content_hash_id` record in `fact_content_daily_performance`. I verify this before aggregating.
- **Table:** `fact_content_daily_performance`, using `month=2026-03`.
- **Time window:** 1–31 March 2026. June 2026 is reserved as a sealed final month and is not used while developing this score.
- **Target / proxy:** a position-adjusted March CTR under-capture score, calculated after aggregation from observed GSC clicks and impressions.
- **Deliberately excluded from features:** `gsc_clicks`, calculated CTR, and any CTR-gap field because they directly form the proxy. Future partitions are also excluded.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Add HF_TOKEN in Colab Secrets and enable Notebook access.")

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# Verification Query 1: raw-table grain
q1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS rows_at_grain
FROM read_parquet('{MARCH}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_violations = con.execute(q1).df()

print("=== Query 1: Raw-table grain check ===")
print("Expected raw grain: report_date × client_hash_id × content_hash_id")
print("Duplicate grain rows found:", len(grain_violations))
display(grain_violations)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Query 1: Raw-table grain check ===
Expected raw grain: report_date × client_hash_id × content_hash_id
Duplicate grain rows found: 0


,client_hash_id,content_hash_id,report_date,rows_at_grain


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Context**

- `client_hash_id`, `content_hash_id`: pseudonymized identifiers used for grouping and later validation, never model features.
- `report_date`: raw daily date used only to define the March window.
- `position_bucket`: comparison context for expected CTR.

**Features**

- `march_impressions`
- `march_avg_position`
- `march_scroll_events`
- `march_sessions_social`
- `observed_days`

**Label / proxy**

- `is_below_position_bucket_median_ctr`: whether an aggregated March candidate has CTR below the median CTR of comparable candidates in its position bucket.
- `march_ctr_gap`: the positive gap between expected bucket CTR and observed March CTR.

**Excluded**

- `gsc_clicks`, `march_ctr`, `expected_ctr_by_position_bucket`, and `march_ctr_gap` are excluded from model features because they define the proxy.
- June 2026 and every later outcome window are excluded during March development.
- Rows where `gsc_data_available IS NOT TRUE` are excluded because they do not contain usable Search Console evidence.

In [2]:
# Verification Query 2: March row count and date span
q2 = f"""
SELECT
    COUNT(*) AS march_daily_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id)) AS march_content_client_candidates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet('{MARCH}')
"""

print("=== Query 2: March slice count and date span ===")
display(con.execute(q2).df())

=== Query 2: March slice count and date span ===


,march_daily_rows,march_content_client_candidates,first_report_date,last_report_date
0,9841378,331437,2026-03-01,2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Five-feature frame

1. **`march_impressions`** — knowable at the decision moment because it is aggregated from eligible March Search Console observations.
2. **`march_avg_position`** — knowable at the decision moment because it is aggregated from eligible March Search Console observations.
3. **`march_scroll_events`** — knowable at the decision moment because it is aggregated from March engagement events.
4. **`march_sessions_social`** — knowable at the decision moment because it is aggregated from March social-session observations.
5. **`observed_days`** — knowable at the decision moment because it counts eligible daily records observed in March.

The CTR proxy is not a feature. It is calculated from GSC clicks and impressions only after aggregation, then kept separate from the five-feature frame.

In [3]:
# Verification Query 3: availability filter
q3 = f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{MARCH}')
),
eligible AS (
    SELECT *
    FROM march
    WHERE gsc_data_available IS TRUE
)
SELECT
    (SELECT COUNT(*) FROM march) AS rows_before_gsc_filter,
    (SELECT COUNT(*) FROM eligible) AS rows_after_gsc_filter,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM eligible)
        / NULLIF((SELECT COUNT(*) FROM march), 0),
        2
    ) AS pct_rows_retained
"""

print("=== Query 3: GSC availability using IS TRUE ===")
display(con.execute(q3).df())

# Build the Lane A monthly feature frame from eligible March records.
feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        COALESCE(scroll_events, 0) AS scroll_events,
        COALESCE(sessions_social, 0) AS sessions_social
    FROM read_parquet('{MARCH}')
    WHERE gsc_data_available IS TRUE
),
monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(scroll_events) AS march_scroll_events,
        SUM(sessions_social) AS march_sessions_social,
        COUNT(*) AS observed_days,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
    FROM daily
    GROUP BY 1, 2
    HAVING
        SUM(gsc_impressions) >= 100
        AND AVG(NULLIF(gsc_avg_position, 0)) IS NOT NULL
),
bucketed AS (
    SELECT *,
        CASE
            WHEN march_avg_position <= 3 THEN '1-3'
            WHEN march_avg_position <= 10 THEN '4-10'
            WHEN march_avg_position <= 20 THEN '11-20'
            WHEN march_avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END AS position_bucket
    FROM monthly
),
scored AS (
    SELECT *,
        MEDIAN(march_ctr) OVER (
            PARTITION BY position_bucket
        ) AS expected_ctr_by_position_bucket
    FROM bucketed
)
SELECT
    client_hash_id,
    content_hash_id,
    position_bucket,
    march_impressions,
    march_avg_position,
    march_scroll_events,
    march_sessions_social,
    observed_days,
    march_ctr,
    expected_ctr_by_position_bucket,
    GREATEST(
        0.0,
        expected_ctr_by_position_bucket - march_ctr
    ) AS march_ctr_gap,
    CASE
        WHEN march_ctr < expected_ctr_by_position_bucket THEN 1
        ELSE 0
    END AS is_below_position_bucket_median_ctr
FROM scored
"""

lane_frame = con.execute(feature_sql).df()

print("=== Five-feature Lane A frame ===")
print("Rows:", len(lane_frame))
display(lane_frame.head(10))

# Deliberate leakage experiment: label copy makes a misleadingly perfect model.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

model_frame = lane_frame.sample(
    n=min(50000, len(lane_frame)),
    random_state=42
).copy()

feature_columns = [
    "march_impressions",
    "march_avg_position",
    "march_scroll_events",
    "march_sessions_social",
    "observed_days",
]

X = model_frame[feature_columns]
y = model_frame["is_below_position_bucket_median_ctr"]

train_index, test_index = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
honest_model.fit(X.loc[train_index], y.loc[train_index])

honest_auc = roc_auc_score(
    y.loc[test_index],
    honest_model.predict_proba(X.loc[test_index])[:, 1]
)

X_leaked = X.copy()
X_leaked["leaked_label_copy"] = y

leaked_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
leaked_model.fit(
    X_leaked.loc[train_index],
    y.loc[train_index]
)

leaked_auc = roc_auc_score(
    y.loc[test_index],
    leaked_model.predict_proba(X_leaked.loc[test_index])[:, 1]
)

print("\n=== Deliberate leakage experiment ===")
print(f"Honest ROC-AUC: {honest_auc:.3f}")
print(f"Leaked ROC-AUC: {leaked_auc:.3f}")
print("The leaked_label_copy column is deliberately removed after this check.")

del X_leaked["leaked_label_copy"]

=== Query 3: GSC availability using IS TRUE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_before_gsc_filter,rows_after_gsc_filter,pct_rows_retained
0,9841378,3611061,36.69


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Five-feature Lane A frame ===
Rows: 101441


,client_hash_id,content_hash_id,position_bucket,march_impressions,march_avg_position,march_scroll_events,march_sessions_social,observed_days,march_ctr,expected_ctr_by_position_bucket,march_ctr_gap,is_below_position_bucket_median_ctr
0,client_9958f0a7ae1df715,content_6c474bcf2783d986,21-50,169.0,28.961381,1.0,0.0,31,0.000000,0.0,0.0,0
1,client_73cda7b4e4f265ea,content_043b99a25eaabc72,21-50,246.0,30.105231,0.0,0.0,31,0.000000,0.0,0.0,0
2,client_23a62021009f63c4,content_8b6382751f9b23d6,21-50,8598.0,36.043364,3.0,0.0,31,0.000814,0.0,0.0,0
3,client_23a62021009f63c4,content_99cc4061cbe916a2,21-50,4161.0,44.627871,163.0,0.0,31,0.000481,0.0,0.0,0
4,client_62f4a7e64f5e0096,content_0bf2def807f81ee1,21-50,383.0,49.420712,0.0,0.0,31,0.000000,0.0,0.0,0
5,client_62f4a7e64f5e0096,content_815c83c85850ac2a,21-50,432.0,22.157857,0.0,0.0,30,0.000000,0.0,0.0,0
6,client_9958f0a7ae1df715,content_a22ef2f4631595f1,21-50,424.0,34.912309,0.0,0.0,31,0.000000,0.0,0.0,0
7,client_9958f0a7ae1df715,content_8a375efe8c1418fb,21-50,103.0,41.753439,0.0,0.0,29,0.000000,0.0,0.0,0
8,client_9958f0a7ae1df715,content_6dfb02b075573e1f,21-50,148.0,48.839990,0.0,0.0,31,0.000000,0.0,0.0,0
9,client_9958f0a7ae1df715,content_6abbb579b216aa96,21-50,200.0,37.586110,0.0,0.0,31,0.005000,0.0,0.0,0



=== Deliberate leakage experiment ===
Honest ROC-AUC: 0.795
Leaked ROC-AUC: 1.000
The leaked_label_copy column is deliberately removed after this check.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- This score uses only March records where `gsc_data_available IS TRUE`; it does not represent content without usable Search Console evidence.
- CTR under-capture is observational. A low position-adjusted CTR can reflect search intent, SERP layout, seasonality, or measurement noise; it does not prove that metadata or content is poor.
- The March window is a single snapshot. It cannot show whether a human review or content change caused later performance improvement.
- This warehouse release contains pseudonymized IDs, not real client names, URLs, titles, or raw search queries. Recommendations must remain human-reviewed and public-safe.

In [4]:
# No fourth verification query: ML-04 requires exactly three verification queries above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.